In [12]:
# Fase 1 — Datos
!pip install kagglehub -q
import kagglehub
import pandas as pd
import numpy as np
import os

path = kagglehub.dataset_download("saurabhbadole/zomato-delivery-operations-analytics-dataset")
archivo = os.listdir(path)[0]
df = pd.read_csv(f'{path}/{archivo}')
print(df.shape)
df.head()

Using Colab cache for faster access to the 'zomato-delivery-operations-analytics-dataset' dataset.
(45584, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,12-02-2022,21:55,22:10,Fog,Jam,2,Snack,motorcycle,3.0,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,13-02-2022,14:55,15:05,Stormy,High,1,Meal,motorcycle,1.0,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,04-03-2022,17:30,17:40,Sandstorms,Medium,1,Drinks,scooter,1.0,No,Metropolitian,21
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,13-02-2022,09:20,09:30,Sandstorms,Low,0,Buffet,motorcycle,0.0,No,Metropolitian,20
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,14-02-2022,19:50,20:05,Fog,Jam,1,Snack,scooter,1.0,No,Metropolitian,41


In [13]:
# Fase 2 — Limpieza
df = df.dropna(subset=['Weather_conditions', 'Road_traffic_density', 'Time_Orderd', 'City', 'multiple_deliveries'])
df['Delivery_person_Age'] = df['Delivery_person_Age'].fillna(df['Delivery_person_Age'].median())
df['Delivery_person_Ratings'] = df['Delivery_person_Ratings'].fillna(df['Delivery_person_Ratings'].median())
df['Festival'] = df['Festival'].fillna('No')

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df['Distance_km'] = haversine(df['Restaurant_latitude'].abs(), df['Restaurant_longitude'].abs(),
                                df['Delivery_location_latitude'].abs(), df['Delivery_location_longitude'].abs())
df = df[(df['Distance_km'] > 0.3) & (df['Distance_km'] < 30)]

def a_minutos(t):
    try:
        h, m = map(int, t.strip().split(':'))
        return h*60 + m
    except: return np.nan

df['t_ordered_min'] = df['Time_Orderd'].apply(a_minutos)
df['t_picked_min'] = df['Time_Order_picked'].apply(a_minutos)
df['Preparation_Time_min'] = df['t_picked_min'] - df['t_ordered_min']
df.loc[df['Preparation_Time_min'] < 0, 'Preparation_Time_min'] += 24*60
df = df.dropna(subset=['Preparation_Time_min'])
df = df[(df['Preparation_Time_min'] >= 0) & (df['Preparation_Time_min'] <= 60)]

def hora_del_dia(t_min):
    h = t_min // 60
    if 6 <= h < 12: return 'Morning'
    if 12 <= h < 18: return 'Afternoon'
    if 18 <= h < 24: return 'Evening'
    return 'Night'

df['Time_of_Day'] = df['t_ordered_min'].apply(hora_del_dia)
print(len(df), 'filas finales')

33752 filas finales


In [14]:
# Fase 3 — Feature engineering
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df['Distancia_al_cuadrado'] = df['Distance_km'] ** 2
trafico_num = df['Road_traffic_density'].map({'Low': 1, 'Medium': 2, 'High': 3, 'Jam': 4})
df['Distancia_x_Trafico'] = df['Distance_km'] * trafico_num

le_weather, le_traffic, le_time, le_vehicle = LabelEncoder(), LabelEncoder(), LabelEncoder(), LabelEncoder()

df['Weather_enc'] = le_weather.fit_transform(df['Weather_conditions'])
df['Traffic_enc'] = le_traffic.fit_transform(df['Road_traffic_density'])
df['Time_enc']    = le_time.fit_transform(df['Time_of_Day'])
df['Vehicle_enc'] = le_vehicle.fit_transform(df['Type_of_vehicle'])

X = df[['Distance_km', 'Distancia_al_cuadrado', 'Distancia_x_Trafico',
        'Weather_enc', 'Traffic_enc', 'Time_enc', 'Vehicle_enc',
        'Preparation_Time_min']]
y = df['Time_taken (min)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(27001, 8) (6751, 8)


In [16]:
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluar(nombre, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{nombre}: MAE={mae:.2f} RMSE={rmse:.2f} R2={r2:.4f}')
    return r2

grid_rf = GridSearchCV(RandomForestRegressor(random_state=42),
    {'n_estimators':[200], 'max_depth':[10,20,None], 'min_samples_split':[2,5]}, cv=3, scoring='r2', n_jobs=-1)
grid_rf.fit(X_train, y_train)
evaluar('Random Forest', y_test, grid_rf.predict(X_test))

Random Forest: MAE=5.26 RMSE=6.66 R2=0.5047


0.5046906234525873

In [17]:
!pip install lightgbm -q
from lightgbm import LGBMRegressor

grid_lgbm = GridSearchCV(LGBMRegressor(random_state=42, verbose=-1),
    {'n_estimators':[200,400], 'max_depth':[4,6,8], 'learning_rate':[0.05,0.1], 'num_leaves':[31,50]}, cv=3, scoring='r2', n_jobs=-1)
grid_lgbm.fit(X_train, y_train)
evaluar('LightGBM', y_test, grid_lgbm.predict(X_test))

LightGBM: MAE=5.24 RMSE=6.64 R2=0.5087


0.5086821875136922

In [18]:
modelo_stacking = StackingRegressor(
    estimators=[('rf', grid_rf.best_estimator_), ('lgbm', grid_lgbm.best_estimator_)],
    final_estimator=Ridge(alpha=1.0), cv=3)
modelo_stacking.fit(X_train, y_train)
y_pred_stacking = modelo_stacking.predict(X_test)
evaluar('Stacking (RF+LGBM)', y_test, y_pred_stacking)

Stacking (RF+LGBM): MAE=5.24 RMSE=6.63 R2=0.5097


0.5096751250785816

In [19]:
# Fase 5 — Exportación
import joblib

joblib.dump(modelo_stacking, 'modelo_final.pkl')
joblib.dump(le_weather, 'le_weather.pkl')
joblib.dump(le_traffic, 'le_traffic.pkl')
joblib.dump(le_time, 'le_time.pkl')
joblib.dump(le_vehicle, 'le_vehicle.pkl')

print('Clases weather:', list(le_weather.classes_))
print('Clases traffic:', list(le_traffic.classes_))
print('Clases time:', list(le_time.classes_))
print('Clases vehicle:', list(le_vehicle.classes_))

Clases weather: ['Cloudy', 'Fog', 'Sandstorms', 'Stormy', 'Sunny', 'Windy']
Clases traffic: ['High', 'Jam', 'Low', 'Medium']
Clases time: ['Afternoon', 'Evening', 'Morning']
Clases vehicle: ['electric_scooter', 'motorcycle', 'scooter']


In [20]:
import zipfile

with zipfile.ZipFile('modelo_deliveryfast.zip', 'w') as zf:
    for f in ['modelo_final.pkl','le_weather.pkl','le_traffic.pkl','le_time.pkl','le_vehicle.pkl']:
        zf.write(f)

from google.colab import files
files.download('modelo_deliveryfast.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>